In [ ]:
pip install -U transformers

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="ibm-granite/granite-3.3-2b-instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
import gradio as gr
import json
import re
from datetime import datetime
from typing import Dict, List, Any, Tuple, Optional
import json5  # Using json5 for more relaxed parsing (allows comments/trailing commas)
from jsonschema import validate, ValidationError
from enum import Enum

# ============================================================================
# CORE ENGINE
# ============================================================================

class CaseStyle(Enum):
    SNAKE_CASE = "snake_case"
    CAMEL_CASE = "camelCase"
    KEBAB_CASE = "kebab-case"
    PASCAL_CASE = "PascalCase"

class JSONRefinerCore:
    def __init__(self):
        self.transformation_history = []

    @staticmethod
    def to_snake_case(text: str) -> str:
        text = re.sub(r'([a-z])([A-Z])', r'\1_\2', text)
        text = re.sub(r'[\s\-]+', '_', text)
        return re.sub(r'[^a-zA-Z0-9_]', '', text).lower()

    @staticmethod
    def to_camel_case(text: str) -> str:
        snake = JSONRefinerCore.to_snake_case(text)
        parts = snake.split('_')
        return parts[0] + ''.join(word.capitalize() for word in parts[1:])

    @staticmethod
    def to_kebab_case(text: str) -> str:
        return JSONRefinerCore.to_snake_case(text).replace('_', '-')

    @staticmethod
    def to_pascal_case(text: str) -> str:
        camel = JSONRefinerCore.to_camel_case(text)
        return camel[0].upper() + camel[1:] if camel else ""

    @classmethod
    def normalize_key(cls, key: str, style: CaseStyle) -> str:
        key = key.strip()
        if style == CaseStyle.SNAKE_CASE: return cls.to_snake_case(key)
        if style == CaseStyle.CAMEL_CASE: return cls.to_camel_case(key)
        if style == CaseStyle.KEBAB_CASE: return cls.to_kebab_case(key)
        if style == CaseStyle.PASCAL_CASE: return cls.to_pascal_case(key)
        return key

    @staticmethod
    def infer_type(value: str) -> Any:
        v = value.strip()
        # Boolean / Null
        val_map = {'true': True, 'false': False, 'yes': True, 'no': False, 'null': None, 'none': None}
        if v.lower() in val_map: return val_map[v.lower()]

        # Numbers
        try:
            if '.' in v: return float(v)
            return int(v)
        except ValueError: pass

        # JSON Structures (Lists/Dicts)
        if (v.startswith('[') and v.endswith(']')) or (v.startswith('{') and v.endswith('}')):
            try: return json5.loads(v)
            except: pass

        return v

    def parse_kv_text(self, text: str, style: CaseStyle) -> Tuple[Dict, List[str]]:
        result, errors = {}, []
        for i, line in enumerate(text.split('\n'), 1):
            line = line.strip()
            if not line or line.startswith('#'): continue
            if ':' not in line:
                errors.append(f"Line {i}: Missing colon")
                continue

            k, v = line.split(':', 1)
            norm_k = self.normalize_key(k, style)
            if norm_k in result:
                errors.append(f"Line {i}: Duplicate key '{norm_k}'")
            result[norm_k] = self.infer_type(v)
        return result, errors

    @staticmethod
    def flatten_json(data: Dict, parent_key: str = '', sep: str = '.') -> Dict:
        items = []
        for k, v in data.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            if isinstance(v, dict):
                items.extend(JSONRefinerCore.flatten_json(v, new_key, sep).items())
            else:
                items.append((new_key, v))
        return dict(items)

    @staticmethod
    def unflatten_json(data: Dict, sep: str = '.') -> Dict:
        result = {}
        for key, value in data.items():
            parts = key.split(sep)
            d = result
            for part in parts[:-1]:
                if part not in d: d[part] = {}
                d = d[part]
            d[parts[-1]] = value
        return result

# ============================================================================
# GRADIO LOGIC
# ============================================================================

core = JSONRefinerCore()
history = []

def process_refine(text, style_name, rm_nulls, do_flatten, pretty):
    try:
        style = CaseStyle(style_name)
        data, errors = core.parse_kv_text(text, style)

        if rm_nulls:
            data = {k: v for k, v in data.items() if v is not None}
        if do_flatten:
            data = core.flatten_json(data)

        out = json.dumps(data, indent=2 if pretty else None, ensure_ascii=False)

        status = "✅ Success" + (f" (with {len(errors)} warnings)" if errors else "")
        stats = f"**Keys:** {len(data)} | **Types:** {list(set(type(v).__name__ for v in data.values()))}"

        history.append({"time": datetime.now().isoformat(), "out": out})
        return out, status, stats
    except Exception as e:
        return "", f"❌ Error: {str(e)}", ""

def process_case_conversion(json_str, target_style_name):
    try:
        data = json5.loads(json_str)
        style = CaseStyle(target_style_name)

        def rec_convert(obj):
            if isinstance(obj, dict):
                return {core.normalize_key(k, style): rec_convert(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [rec_convert(i) for i in obj]
            return obj

        converted = rec_convert(data)
        return json.dumps(converted, indent=2), "✅ Converted"
    except Exception as e:
        return "", f"❌ Error: {str(e)}"

# ============================================================================
# UI LAYOUT
# ============================================================================

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎯 JSON Refiner Pro")

    with gr.Tabs():
        with gr.TabItem("🔧 Refine"):
            with gr.Row():
                with gr.Column():
                    inp = gr.Textbox(label="KV Input", lines=10, placeholder="key: value")
                    style_opt = gr.Radio(choices=[e.value for e in CaseStyle], value="snake_case", label="Case")
                    with gr.Row():
                        f1 = gr.Checkbox(label="Remove Nulls")
                        f2 = gr.Checkbox(label="Flatten")
                        f3 = gr.Checkbox(label="Pretty Print", value=True)
                    btn = gr.Button("Transform", variant="primary")
                with gr.Column():
                    out_json = gr.Code(label="JSON Output", language="json")
                    out_status = gr.Markdown()
                    out_stats = gr.Markdown()

            btn.click(process_refine, [inp, style_opt, f1, f2, f3], [out_json, out_status, out_stats])

        with gr.TabItem("🎨 Case Converter"):
            with gr.Row():
                case_in = gr.Code(label="Input JSON", language="json", lines=10)
                case_style = gr.Radio(choices=[e.value for e in CaseStyle], value="camelCase", label="Target")
            case_btn = gr.Button("Convert Keys")
            case_out = gr.Code(label="Result", language="json")
            case_btn.click(process_case_conversion, [case_in, case_style], [case_out, gr.Markdown()])

        with gr.TabItem("📜 History"):
            hist_display = gr.Textbox(label="Session Logs", lines=15)
            hist_btn = gr.Button("Refresh History")
            hist_btn.click(lambda: "\n---\n".join([f"{h['time']}\n{h['out']}" for h in history]), None, hist_display)

if __name__ == "__main__":
    demo.launch()

In [ ]:
!pip install -q json5 jsonschema gradio